In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import joblib

In [2]:
train_df = pd.read_csv("../data/train.csv", index_col="ID")
clients_df = pd.read_csv("../data/clientes.csv", index_col="customer_id")
test_df = pd.read_csv("../data/test.csv")

# <font color='#E8800A'>3 · Explore the data</font> <a class="anchor" id="explore"></a>

1. Sort columns by row
2. Count repeated rows
3. Look at the target
4. Locate missing values
5. Describe numeric columns and their shape
6. Check for impossible values
7. Test wether the missingness is random
8. Plot missing values and outliers

In [11]:
train_df.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000000,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448
10000001,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510
10000002,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730
10000003,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,1.448402
10000004,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,-1.058890


In [35]:
identifiers = ["customer_id"]
noise = ["random_noise"]
target = "repeat_purchase_90d"

numeric = [c for c in train_df.columns
           if c not in identifiers + noise + [target]
           and pd.api.types.is_numeric_dtype(train_df[c])]
categorical = [c for c in train_df.columns
               if c not in identifiers + noise + [target] + numeric]

In [36]:
low_cardinality = {c: train_df[c].nunique() for c in numeric if train_df[c].nunique() <= 10}
levels = train_df[categorical].nunique().sort_values(ascending=False)
print(f"Numeric columns with low cardinality: {low_cardinality}")
for column in levels[levels <= 4].index:
    print(column, sorted(train_df[column].dropna().unique()))

Numeric columns with low cardinality: {'invoice_count_current_purchase': 5}
document_series ['Ctrl Z Series 1', 'Ctrl Z Series 2', 'Ctrl Z Series 3']
warehouse ['Final Excel Warehouse 1', 'Final Excel Warehouse 3']


There are no inconsistencies in these values

In [37]:
train_df["invoice_count_current_purchase"].value_counts().sort_index()

invoice_count_current_purchase
1.0    6116
2.0     312
3.0      14
4.0      21
5.0       6
Name: count, dtype: int64

This column represents a count so we keep it as numeric

In [38]:
print(f"Exact duplicate rows: {train_df.duplicated().sum()}")
print(f"Surplus rows per entity: {train_df.duplicated(subset="customer_id").sum()}")

Exact duplicate rows: 10
Surplus rows per entity: 5924


In [39]:
count_target = train_df['repeat_purchase_90d'].value_counts()

target_prop = count_target.to_frame("count").assign(proportion=lambda d: d["count"] / d["count"].sum())
target_prop

,count,proportion
repeat_purchase_90d,,
1,3654,0.559229
0,2880,0.440771


In [40]:
rows_with_null = train_df.isna().any(axis=1)
per_column = train_df.isna().sum()

pd.Series({
    "rows with a null": f"{rows_with_null.sum():,} of {len(train_df):,}"
                        f" ({100 * rows_with_null.mean():.2f}%)",
    "columns with a null": f"{(per_column > 0).sum()} of {train_df.shape[1]}",
    "null cells": f"{train_df.isna().sum().sum():,}",
    "worst column": f"{per_column.idxmax()} ({per_column.max()} nulls)",
}).to_frame("missing values as the file arrives")

,missing values as the file arrives
rows with a null,"2,605 of 6,534 (39.87%)"
columns with a null,36 of 38
null cells,"9,818"
worst column,prior_std_gap_days (1478 nulls)


In [42]:
summary = train_df[numeric].describe().T
shape = summary.copy()
shape["mean/median"] = shape["mean"] / shape["50%"].replace(0, np.nan)
shape["skew"] = train_df[numeric].skew()
shape["% zeros"] = 100 * (train_df[numeric] == 0).mean()

shape[["mean", "50%", "mean/median", "skew", "% zeros"]].round(2)

,mean,50%,mean/median,skew,% zeros
Total a Pagar,925.56,476.74,1.94,4.21,0.00
Total Bruto,886.15,445.74,1.99,4.31,0.00
Total Liquido,773.34,389.91,1.98,4.15,0.00
Total Imp_,152.22,79.74,1.91,4.84,3.26
Total Portes,1.15,0.00,NaN,19.55,82.89
Total Desc_ Global,0.35,0.00,NaN,27.46,97.32
Total Desc_ Linha,112.47,13.13,8.57,5.92,37.36
Custo,328.17,152.54,2.15,4.61,7.44
N_ Detalhes,21.52,11.00,1.96,2.98,0.00
invoice_count_current_purchase,1.07,1.00,1.07,6.44,0.00
